# 🔍 Notebook 03: Phân Tích Lỗi Chi Tiết (Error Analysis)
**Đề tài:** Phân tích cảm xúc theo khía cạnh trên UIT-ViSFD.

Notebook này tập trung mổ xẻ các lỗi dự đoán của mô hình theo **3 thử thách trọng tâm** của đề tài:
1. **Câu đa khía cạnh (Multi-aspect):** Tỷ lệ chính xác khi câu có $\ge 2$ khía cạnh.
2. **Cảm xúc trái chiều (Conflicting Sentiment):** Khả năng phân loại khi câu vừa khen bộ phận này vừa chê bộ phận khác.
3. **Lớp vắng mặt (Absent / Empty Class):** Khả năng nhận diện câu không nhắc đến khía cạnh nào (`{OTHERS}`).
4. **Phân loại dạng lỗi:** False Positive (Đoán thừa), False Negative (Bỏ sót), và Polarity Mismatch (Đúng khía cạnh nhưng sai cực tính).

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, "..")

from src.data.load import load_split
from src.data.schema import ASPECTS, POLARITES

sns.set_theme(style="whitegrid")

## 1. Ghép Gold Dev với Predictions Dev để phân tích lỗi

In [ ]:
dev_gold = load_split(Path("../data/raw"), "dev")
preds_path = Path("../outputs/predictions/baseline_dev.jsonl")

with open(preds_path, "r", encoding="utf-8") as f:
    dev_preds = [json.loads(line) for line in f]

records = []
for g, p in zip(dev_gold, dev_preds):
    gold_pairs = g["pairs"]
    pred_pairs = {item["aspect"]: item["sentiment"] for item in p["pred"]}
    
    gold_aspects = set(gold_pairs.keys())
    pred_aspects = set(pred_pairs.keys())
    
    tp_aspects = gold_aspects & pred_aspects
    fp_aspects = pred_aspects - gold_aspects
    fn_aspects = gold_aspects - pred_aspects
    
    # Kiểm tra cực tính trên các khía cạnh đúng
    correct_joint = [a for a in tp_aspects if gold_pairs[a] == pred_pairs[a]]
    wrong_sentiment = [a for a in tp_aspects if gold_pairs[a] != pred_pairs[a]]
    
    records.append({
        "id": g["id"],
        "text": g["text"],
        "n_gold_aspects": len(gold_aspects),
        "is_empty_gold": len(gold_aspects) == 0,
        "is_empty_pred": len(pred_aspects) == 0,
        "has_conflicting_gold": ("Positive" in gold_pairs.values()) and ("Negative" in gold_pairs.values()),
        "fp_aspects": list(fp_aspects),
        "fn_aspects": list(fn_aspects),
        "wrong_sentiment": wrong_sentiment,
        "correct_joint": correct_joint,
        "is_fully_correct": (gold_pairs == pred_pairs)
    })

df_errors = pd.DataFrame(records)
print(f"Tổng số mẫu Dev: {len(df_errors)}")
print(f"Số câu đoán đúng hoàn toàn 100%: {df_errors['is_fully_correct'].sum()} ({df_errors['is_fully_correct'].mean():.2%})")

## 2. Phân tích độ chính xác theo Số lượng khía cạnh trong câu (1 vs 2 vs 3+)

In [ ]:
df_errors["aspect_bin"] = df_errors["n_gold_aspects"].apply(lambda n: "0 (Rỗng)" if n == 0 else ("1" if n == 1 else ("2" if n == 2 else "3+")))
acc_by_bin = df_errors.groupby("aspect_bin")["is_fully_correct"].agg(["count", "mean"])
acc_by_bin.columns = ["Số mẫu", "Tỷ lệ đúng hoàn toàn"]
print(acc_by_bin)

## 3. Phân tích trên tập câu Cảm xúc trái chiều (Conflicting Sentiments)

In [ ]:
conflict_df = df_errors[df_errors["has_conflicting_gold"]]
print(f"Số câu có cảm xúc trái chiều trên Dev: {len(conflict_df)}")
print(f"Tỷ lệ đúng hoàn toàn ở các câu trái chiều: {conflict_df['is_fully_correct'].mean():.2%}")

# In một số câu trái chiều bị sai
wrong_conflicts = conflict_df[~conflict_df["is_fully_correct"]].head(3)
for _, row in wrong_conflicts.iterrows():
    print("-" * 60)
    print(f"Câu: {row['text']}")
    print(f"Đúng nhưng sai cảm xúc : {row['wrong_sentiment']}")
    print(f"Bỏ sót khía cạnh       : {row['fn_aspects']}")
    print(f"Đoán thừa khía cạnh    : {row['fp_aspects']}")

## 4. Phân tích xử lý lớp vắng mặt (Các câu không có khía cạnh nào)

In [ ]:
empty_gold = df_errors[df_errors["is_empty_gold"]]
print(f"Số mẫu Gold rỗng trên Dev: {len(empty_gold)}")
print(f"Số mẫu được dự đoán đúng là rỗng: {empty_gold['is_empty_pred'].sum()} ({empty_gold['is_empty_pred'].mean():.2%})")